# 03 — S3 Tables (Apache Iceberg)
**Objective:** native Iceberg tables on ObjectScale via Spark's REST catalog — no data movement.

**Customer sees:** table bucket + namespace, DDL/DML, MERGE, schema evolution, time travel, and a read-only denial.
**TC:** TC3.1–TC3.7

## Step 1 — Session
SparkSession with the `os` REST catalog pointed at the ObjectScale TableHead endpoint (SigV4, table bucket ARN as prefix). The catalog identity is passed explicitly per session.

In [ ]:
import pandas as pd
from poc import config, results, s3util, viz
import poc.spark_helpers as sh

spark = sh.build_session("obs-poc", tables_user="tables-rw")
viz.table(pd.DataFrame(sh.versions(spark).items(), columns=["component", "version"]))
print("run id:", results.run_id(), "| session identities:", sh.identities(spark))
print("catalog os ->", config.ICEBERG_REST_URI, "| warehouse", config.ICEBERG_WAREHOUSE)
T = f"os.{config.TABLES_NAMESPACE}.orders_ice"

**Presenter notes** (hidden)

Catalog `os`: Iceberg REST on the TableHead endpoint (HTTPS 19021), warehouse = the poc-lakehouse table bucket ARN.

## Step 2 — Table bucket & namespace via S3 Tables API (TC3.1)
Confirm the table bucket and namespace through the AWS-compatible API and show the responses.

In [ ]:
st = s3util.s3tables("tables-rw")
arn = config.table_bucket_arn()
tbl = st.list_table_buckets()["tableBuckets"]
namespaces = st.list_namespaces(tableBucketARN=arn).get("namespaces", [])
viz.table(pd.DataFrame(tbl)); viz.table(pd.DataFrame(namespaces)[["namespace"]])
has_bucket = any(b.get("name") == config.TABLE_BUCKET for b in tbl)
has_ns = any(n["namespace"] == [config.TABLES_NAMESPACE] for n in namespaces)
res = results.write("TC3.1", "bucket+namespace", has_bucket and has_ns,
                    {"table_bucket": config.TABLE_BUCKET, "found": has_bucket,
                     "namespace": config.TABLES_NAMESPACE, "ns_found": has_ns})
viz.banner(res["status"], "TC3.1",
           f"bucket={config.TABLE_BUCKET} ns={config.TABLES_NAMESPACE}")

**Presenter notes** (hidden)

boto3 s3tables client + x-emc-namespace header.

## Step 3 — SHOW NAMESPACES + CREATE TABLE (TC3.2)
The same objects are visible through Spark's Iceberg catalog — one control plane, no ETL.

In [ ]:
spark.sql("SHOW NAMESPACES IN os").show()
spark.sql(f"DROP TABLE IF EXISTS {T}")
spark.sql(f"""CREATE TABLE {T}
 (order_id string, customer string, status string, amount double, qty int)
 USING iceberg""")
created = "orders_ice" in [r["tableName"] for r in
                           spark.sql(f"SHOW TABLES IN os.{config.TABLES_NAMESPACE}").collect()]
res = results.write("TC3.2", "create table", created, {"table": T})
viz.banner(res["status"], "TC3.2", f"{T} created")

**Presenter notes** (hidden)

SparkCatalog 'os' — type=rest, SigV4-signed.

## Step 4 — INSERT + SELECT (TC3.3)
Insert rows through the Iceberg catalog; they land in the managed `--table-s3` bucket on ObjectScale.

In [ ]:
spark.sql(f"""INSERT INTO {T}
 VALUES ('o1','cust-001','PAID',199.0,2),('o2','cust-002','NEW',49.5,1),
        ('o3','cust-003','SHIPPED',1200.0,10)""")
rows = spark.sql(f"SELECT * FROM {T} ORDER BY order_id").toPandas()
viz.table(rows)
res = results.write("TC3.3", "insert+select", len(rows) == 3, {"rows": len(rows)})
viz.banner(res["status"], "TC3.3", f"{len(rows)} rows")

**Presenter notes** (hidden)

Data files are visible in the OBS UI under <table-uuid>--table-s3.

## Step 5 — MERGE INTO (TC3.4)
Upsert a changed row — Iceberg's row-level update over object storage, before and after side by side.

In [ ]:
before = rows.copy()
spark.sql(f"""MERGE INTO {T} t
 USING (SELECT 'o2' order_id,'cust-002' customer,'PAID' status,55.0 amount,1 qty) u
 ON t.order_id=u.order_id WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *""")
after = spark.sql(f"SELECT * FROM {T} ORDER BY order_id").toPandas()
viz.table(before.assign(state="before")); viz.table(after.assign(state="after"))
merged = (len(after) == 3
          and after.loc[after.order_id == "o2", "status"].iloc[0] == "PAID")
res = results.write("TC3.4", "merge", merged, {"rows_after": len(after)})
viz.banner(res["status"], "TC3.4", "o2 NEW -> PAID")

**Presenter notes** (hidden)

MERGE INTO needs the Iceberg SQL extensions — already configured.

## Step 6 — Schema evolution (TC3.5)
Add a column without rewriting data.

In [ ]:
spark.sql(f"ALTER TABLE {T} ADD COLUMNS (region string)")
cols = spark.sql(f"DESCRIBE {T}").toPandas()
viz.table(cols)
res = results.write("TC3.5", "add column", "region" in set(cols.col_name))
viz.banner(res["status"], "TC3.5", "column region added")

**Presenter notes** (hidden)

Metadata-only operation — instant.

## Step 7 — Time travel & snapshot history (TC3.6)
Query the first snapshot vs the current state, and show the snapshot log.

In [ ]:
snaps = spark.sql(f"SELECT snapshot_id, committed_at FROM {T}.snapshots ORDER BY committed_at").toPandas()
old = spark.sql(f"SELECT status FROM {T} VERSION AS OF {snaps.snapshot_id.iloc[0]} WHERE order_id='o2'").toPandas()
cur = spark.sql(f"SELECT status FROM {T} WHERE order_id='o2'").toPandas()
viz.table(snaps)
viz.table(pd.concat([old.assign(view="first snapshot"), cur.assign(view="current")]))
travelled = (len(snaps) >= 2 and old.status.iloc[0] == "NEW"
             and cur.status.iloc[0] == "PAID")
res = results.write("TC3.6", "time travel", travelled,
                    {"snapshots": len(snaps), "o2_first": old.status.iloc[0],
                     "o2_current": cur.status.iloc[0]})
viz.banner(res["status"], "TC3.6",
           f"{len(snaps)} snapshots; o2 was {old.status.iloc[0]}, is {cur.status.iloc[0]}")

**Presenter notes** (hidden)

VERSION AS OF uses snapshot_id; TIMESTAMP AS OF is also supported.

## Step 8 — Read-only denial (TC3.7)
One full, valid INSERT is run twice. As **tables-rw** it must succeed (positive control). Then the session is rebuilt with the **tables-ro** identity: SELECT must still work and the *same* INSERT must fail with an access denial. PASS only if the error really is a permission error and no row was added.

In [ ]:
INSERT = f"INSERT INTO {T} VALUES ('o9','cust-009','NEW',10.0,1,'EU')"
count = lambda s: s.sql(f"SELECT count(*) FROM {T}").first()[0]

n0 = count(spark)
spark.sql(INSERT)                                  # positive control: tables-rw
n1 = count(spark)
control_ok = n1 == n0 + 1
spark.stop()

spark = sh.build_session("obs-poc-ro", tables_user="tables-ro")
ident = sh.identities(spark)
try:
    ro_rows, ro_select_error = count(spark), None
except Exception as e:
    ro_rows, ro_select_error = None, str(e).splitlines()[0][:200]
try:
    spark.sql(INSERT)
    inserted, denial, first_line = True, None, "INSERT succeeded"
except Exception as e:
    inserted, denial = False, sh.access_denied_reason(e)
    first_line = str(e).strip().splitlines()[0][:200]
n2 = count(spark) if ro_select_error is None else None

if inserted or (n2 is not None and n2 != n1):
    st37, why37 = "FAIL", f"the read-only identity was able to write ({n1} -> {n2} rows)"
elif ident["tables"] != "tables-ro":
    st37, why37 = "INCONCLUSIVE", f"session ran as {ident['tables']}, not tables-ro"
elif not control_ok:
    st37, why37 = "INCONCLUSIVE", "positive control failed: the INSERT did not succeed as tables-rw"
elif ro_select_error:
    st37, why37 = "INCONCLUSIVE", f"tables-ro could not even read: {ro_select_error}"
elif not denial:
    st37, why37 = "INCONCLUSIVE", f"INSERT failed, but not with an access denial: {first_line}"
else:
    st37, why37 = "PASS", f"tables-ro reads {ro_rows} rows; INSERT denied: {denial}"
res = results.write("TC3.7", "read-only identity denied", status=st37, detail=why37,
    metrics={"identity": ident["tables"], "identity_key": ident["tables_key"],
             "rw_control_insert_ok": control_ok, "ro_select_rows": ro_rows,
             "rows_before_ro_insert": n1, "rows_after_ro_insert": n2,
             "ro_insert_error": denial or first_line})
viz.table(pd.DataFrame([
    {"identity": "tables-rw", "statement": "INSERT", "result": "OK" if control_ok else "FAILED"},
    {"identity": ident["tables"], "statement": "SELECT count(*)",
     "result": ro_rows if ro_select_error is None else ro_select_error},
    {"identity": ident["tables"], "statement": "INSERT", "result": denial or first_line}]))
viz.banner(st37, "TC3.7 read-only denial", why37)

**Presenter notes** (hidden)

tables-ro: S3TablesReadOnlyAccess + ECSS3ReadOnlyAccess. The identity is switched through catalog properties (rest.access-key-id / s3.access-key-id), not environment variables, and build_session verifies the running session really has it.

## Step 9 — List tables via API
Back to the control plane: list tables in the namespace via the S3 Tables API.

In [ ]:
tabs = st.list_tables(tableBucketARN=arn, namespace=config.TABLES_NAMESPACE)["tables"]
viz.table(pd.DataFrame(tabs)[["name", "type"]])
print(f"{len(tabs)} table(s) in {config.TABLES_NAMESPACE}")
spark.stop()

**Presenter notes** (hidden)

The same table list is visible in the OBS UI under S3 Tables.